Note: I removed my API key from this document so the notebook will not run without an API key being defined.

RQ1 - How many wine shops are in Denver?

- Using the Google Maps API to collect a starting point of wine retailers in Denver, CO
- Focusing on wine retailers, specifically. Not bars/restaurants that serve wine - as the premise is "wine that I can buy a bottle of (and take home)"

Google Maps API Docs: https://developers.google.com/maps/documentation/places/web-service

In [ ]:
%pip install --quiet python-dotenv

In [ ]:
#setup 
import pandas as pd
import requests
import numpy as np
import os
from dotenv import load_dotenv

api_key = os.getenv('google_api_key')
my_address = os.getenv('my_address')

Finding the liquor stores in Denver:

- Denver Metro Area "center" = 39.742043 lat, -104.991531 lon per https://www.latlong.net/place/denver-co-usa-3174.html
- Denver spans ~154 sq miles per https://web.archive.org/web/20230305171359/http://www.usa.com/denver-co.htm

In [ ]:
#Lat/lon of my address
# This cell is intentionally redacted and will not print a home address.
my_coords = [None, None]
my_coords

In [ ]:
df_denver_liquor = pd.read_csv("denver_liquor-licenses.csv")

In [ ]:
df_denver_liquor.columns

In [ ]:
df_denver_liquor.shape

In [ ]:
df_denver_liquor_cleaned = df_denver_liquor.dropna(axis=1, how='all')

In [ ]:
df_denver_liquor_cleaned.columns

In [ ]:
df_denver_liquor_cleaned['LICENSES'].value_counts()

# Type of Liquor Licenses:
source: https://content.leg.colorado.gov/sites/default/files/2022_update_liquor_handbook_with_cover.pdf 

### Off-premises retail licenses and permits.
These licensed retailers may sell alcohol in original sealed containers for off-premises consumption.

### On-premises retail licenses and permits.
Establishments that sell alcohol for consumption on the licensed premises.

### Hotel and restaurant license.
These licenses are issued to hotels and restaurants selling alcohol beverages to customers.

### Lodging and entertainment facility license.
These licenses are issued to either a lodging facility, wherethe primary business is to provide the public with sleeping rooms and meeting facilities, or an
entertainment facility, where the primary business is to provide the public with sports or entertainment activities. 

### Optional premises license.
These licenses are issued to an outdoor sports and recreational facility

### Tastings licenses.
An alcohol beverage tastings permit allows a retail liquor store or liquor-licensed drugstore to conduct sample tastings of alcohol within their establishment. Tastings are not to exceed a total of 5 hours per day and they can only be held between the hours of 11 a.m. to 9 p.m.

https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Business-Licensing/Business-licenses/Liquor/Tastings-permit

## Methodology notes on license filtering

For the purposes of this project, I am filtering for licenses that are 'LIQUOR - STORE' or 'LIQUOR - TASTINGS'. 'LIQUOR - STORE' license types indicate they are able to sell alcohol for off-premises consumption. In the case a retail liquor store has a 'LIQUOR - TASTINGS' license, they are represented as license type, 'LIQUOR- TASTINGS' in this data table.

In [ ]:
df_denver_liquor_stores = df_denver_liquor_cleaned[df_denver_liquor_cleaned['LICENSES'].isin(['LIQUOR - STORE', 'LIQUOR - TASTINGS'])]

In [ ]:
df_denver_liquor_stores

In [ ]:
import regex
df_denver_liquor_stores_active = df_denver_liquor_stores[df_denver_liquor_stores['LIC_STATUS'].str.contains('ACTIVE', case=False)]

In [ ]:
df_denver_liquor_stores_active['LIC_STATUS'].unique()

In [ ]:
len(df_denver_liquor_stores_active)

In [ ]:
df_denver_liquor_stores_active.head()

In [ ]:
df_denver_liquor_stores_active_clean.to_csv('denver_liquor_stores.csv')

In [ ]:
df_denver_liquor_stores_active.columns

In [ ]:
df_denver_liquor_stores_active_clean = df_denver_liquor_stores_active.drop(columns = [
    'OBJECTID', 'BFN', 'HEARING_DATE', 'HEARING_TIME', 'HEARING_STATUS', 'HEARING_TIMESTAMP'
]
    
)

## Liquor store location data

This dataset contains several markers for location data, including:
- Full address
- A "Global ID"
- 'x' - x coordinates
- 'y' - y coordinates

### Colorado government use of 'x' and 'y' coordinates

Colorado government data is provided using Colorado State Plane Coordinates. To get lat/lon, I can either convert the 'x' and 'y' accordingly, or use the address geolocator.


In [ ]:
#Lat/lon of my address
# This cell is intentionally redacted and will not print a home address.
my_coords = [None, None]
my_coords

pyproj to transform Colorado State Plane to lat/lon: https://pyproj4.github.io/pyproj/stable/api/transformer.html#pyproj.transformer.Transformer.transform

In [ ]:
%pip install pyproj

In [ ]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:2232", "EPSG:4326", always_xy=True)

df_denver_liquor_stores_active_clean['lon'], df_denver_liquor_stores_active_clean['lat'] = transformer.transform(
    df_denver_liquor_stores_active_clean['x'].values,
    df_denver_liquor_stores_active_clean['y'].values
)

print(df_denver_liquor_stores_active_clean[['x', 'y', 'lon', 'lat']].head(3))

In [ ]:
df_denver_liquor_stores_active_clean.head()

In [ ]:
df_denver_liquor_stores_active_clean.to_csv('denver_liquor_stores.csv', index=False)

## Calculating walking distance
Using the Google Maps API to calculate walking distance between each Denver liquor store and my home address.
Then filtering for those where the walking distance is =<20 minutes -- deciding this is a reasonable "walking distance" for a 'portal' to Sonoma.

Docs for Google Maps Distance Matrix API: https://developers.google.com/maps/documentation/distance-matrix/start

### First - get a clean dataframe that has the addresses and coordinates of both the liquor stores in Denver and my home

In [ ]:
#Create a df with my address and all liquor stores

df_denver_liquor_stores_address = df_denver_liquor_stores_active_clean[['BUS_PROF_NAME','ADDRESS_LINE1','CITY','STATE','lon','lat']]
df_denver_liquor_stores_address.head()

In [ ]:
#add my address
my_address = {
    'BUS_PROF_NAME': np.nan, 
    'ADDRESS_LINE1': my_address,
    'CITY': 'Denver',
    'STATE': 'CO',
    'lon': my_coords[1], 
    'lat': my_coords[0]
    }

In [ ]:
df_my_address = pd.DataFrame([my_address])

In [ ]:
dfs_to_concat = [df_denver_liquor_stores_address, df_my_address]
df_all_address = pd.concat(dfs_to_concat)

In [ ]:
df_all_address

In [ ]:
print(df_all_address.shape)
print(df_all_address['ADDRESS_LINE1'].notna().sum())
print(df_all_address['ADDRESS_LINE1'].head(10))

In [ ]:
duration_rows = []
addresses = []

for index, row in df_all_address.iterrows():
    address = row["ADDRESS_LINE1"]
    addresses.append(address)

    if pd.isna(row['ADDRESS_LINE1']):
        continue
    else:
        address_formatted = row['ADDRESS_LINE1'].replace(' ', '+').replace('#','')
        address_param = f"{address_formatted},+Denver,+CO"
        url = f"https://maps.googleapis.com/maps/api/distancematrix/json?destinations={address_param}&origins=39.728886%2C-104.9715073&mode=walking&key={api_key}"
    response = requests.get(url)
    data = response.json()

    for row_data in data.get("rows", []):
        for element in row_data.get("elements", []):
            duration_rows.append({
                "ADDRESS_LINE1": address,
                "status": element.get("status"),
                "duration_text": element.get("duration", {}).get("text"),
                "duration_value": element.get("duration", {}).get("value"),
            })

In [ ]:
df_duration = pd.DataFrame(duration_rows)

In [ ]:
df_duration.to_csv('duration_data.csv')

In [ ]:
df_duration.head()

In [ ]:
df_duration.dtypes

In [ ]:
df_all_address.dtypes

In [ ]:
from fuzzymatcher import fuzzy_left_join

df_all_address_durations = fuzzy_left_join(
    df_all_address,
    df_duration,
    left_on=["ADDRESS_LINE1"],
    right_on=["ADDRESS_LINE1"]
)

In [ ]:
df_all_address_durations.to_csv('denver_addresses_durations.csv')

In [ ]:
df_all_address_durations.head()

'duration_value' represents duration in seconds. For this, I am looking for walks that are 15 minutes or less.
15 minutes = 900 seconds

In [ ]:
df_denver_liquor_stores_in_walking_distance = df_all_address_durations[df_all_address_durations['duration_value']<=900]

In [ ]:
df_denver_liquor_stores_in_walking_distance

In [ ]:
df_denver_liquor_stores_in_walking_distance.to_csv('denver_liquor_stores_walking_distance.csv')